# SEU — Sampled Expected Utility

Implements all acquisition strategies:
- **SEU-US**: uniform random candidate sampling
- **SEU-ES**: error/uncertainty-based candidate sampling
- **SEU-Accuracy**: ablation — accuracy gain instead of log-gain
- **Uniform baseline**: random acquisition

v2 changes: `sample_fraction=1.0` (full candidate scoring), `verbose` debug option.

In [20]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath(".."), "src"))

import numpy as np
from naive_bayes import NaiveBayesCategorical

## Utility Functions

**Log-gain** (primary utility): $U = \log P(y_i \mid x_i^{\text{new}}) - \log P(y_i \mid x_i^{\text{old}})$

**Accuracy gain** (ablation): $U = \mathbf{1}[\hat{y}^{\text{new}} = y_i] - \mathbf{1}[\hat{y}^{\text{old}} = y_i]$

In [21]:
def log_gain(proba_before, proba_after, y_val, eps=1e-15):
    """Log-gain utility for a single instance."""
    p_before = np.clip(proba_before[y_val], eps, 1.0)
    p_after  = np.clip(proba_after[y_val],  eps, 1.0)
    return np.log(p_after) - np.log(p_before)

def accuracy_gain(pred_before, pred_after, y_val):
    """Accuracy gain (0/1) for a single instance."""
    return float(pred_after == y_val) - float(pred_before == y_val)

## SEU Score

$$E(q_{i,j}) = \frac{1}{C_{i,j}} \sum_k U(F_{i,j} = v_k) \cdot P(F_{i,j} = v_k)$$

In [22]:
def compute_seu_score(model, X_masked, y, i, j, utility="log_gain",
                      acquisition_cost=1.0):
    """
    Compute SEU score for acquiring feature j of instance i.
    Returns float score.
    """
    vals, val_probs = model.feature_value_proba(X_masked[i], j)
    if len(vals) == 0:
        return 0.0

    proba_before = model.predict_proba(X_masked[i:i+1])[0]
    pred_before  = model.classes_[np.argmax(proba_before)]

    expected_utility = 0.0
    for v_k, p_k in zip(vals, val_probs):
        row_hyp = X_masked[i].copy()
        row_hyp[j] = v_k
        proba_after = model.predict_proba(row_hyp[None, :])[0]
        pred_after  = model.classes_[np.argmax(proba_after)]

        if utility == "log_gain":
            u = log_gain(proba_before, proba_after, y[i])
        else:
            u = accuracy_gain(pred_before, pred_after, y[i])

        expected_utility += u * p_k

    return expected_utility / acquisition_cost

## Candidate Sampling Strategies

In [23]:
def select_candidates_us(missing_entries, n_sample, rng):
    """SEU-US: uniform random sampling of candidate (i, j) pairs."""
    if len(missing_entries) <= n_sample:
        return missing_entries
    idx = rng.choice(len(missing_entries), size=n_sample, replace=False)
    return [missing_entries[k] for k in idx]

def select_candidates_es(missing_entries, model, X_masked, n_sample, rng):
    """
    SEU-ES: sample proportional to prediction uncertainty (1 - max_proba),
    prioritizing misclassified or uncertain instances.
    """
    if len(missing_entries) <= n_sample:
        return missing_entries

    instances = list({i for i, j in missing_entries})
    proba = model.predict_proba(X_masked[instances])
    uncertainty = 1.0 - proba.max(axis=1)
    inst_unc = dict(zip(instances, uncertainty))

    scores = np.array([inst_unc[i] for i, j in missing_entries]) + 1e-6
    probs  = scores / scores.sum()
    idx = rng.choice(len(missing_entries), size=n_sample, replace=False, p=probs)
    return [missing_entries[k] for k in idx]

## Acquisition Loop

Key parameters:
- `sample_fraction=1.0` — score all candidates (no subsampling, removes noise)
- `verbose=True` — log per-round utility distribution (mean/min/max) to `debug_file`

In [24]:
def run_acquisition(X_full, y, missing_rate, strategy, seed=0,
                    acquisition_cost=1.0, sample_fraction=1.0,
                    utility="log_gain", verbose=False, debug_file=None):
    """
    Run one full acquisition experiment.

    Parameters
    ----------
    X_full          : np.ndarray (n, d)  fully observed encoded features
    y               : np.ndarray (n,)   class labels
    missing_rate    : float
    strategy        : str  'uniform' | 'seu-us' | 'seu-es' | 'seu-accuracy'
    seed            : int
    acquisition_cost: float  cost per feature acquisition
    sample_fraction : float  fraction of missing entries scored per SEU round
                             (1.0 = full scoring, no candidate subsampling)
    utility         : str    'log_gain' | 'accuracy' (overridden for seu-accuracy)
    verbose         : bool   if True, log per-round utility stats to debug_file
    debug_file      : str    path to write verbose log (default: utility_debug.txt)

    Returns
    -------
    cost_curve     : list of cumulative costs at each acquisition step
    accuracy_curve : list of validation accuracy after each acquisition
    """
    from data_utils import apply_mask
    import pandas as pd

    rng = np.random.default_rng(seed)
    X_masked, mask = apply_mask(pd.DataFrame(X_full), missing_rate, rng)
    X_masked = X_masked.values.copy()

    n, d = X_masked.shape
    missing_entries = [(i, j) for i in range(n)
                       for j in range(d) if mask[i, j]]

    # Fixed 80/20 train/val split
    rng_split = np.random.default_rng(seed + 9999)
    idx = rng_split.permutation(n)
    train_idx, val_idx = idx[:int(0.8 * n)], idx[int(0.8 * n):]

    if strategy == "seu-accuracy":
        utility = "accuracy"
        strategy_base = "seu-us"
    else:
        strategy_base = strategy

    # sample_fraction=1.0 means score all candidates
    n_sample = max(1, int(len(missing_entries) * sample_fraction))

    # Train initial model on masked data
    model = NaiveBayesCategorical()
    model.fit(X_masked[train_idx], y[train_idx])
    acc = (model.predict(X_masked[val_idx]) == y[val_idx]).mean()

    cost_curve     = [0.0]
    accuracy_curve = [acc]
    cumulative_cost = 0.0
    remaining = list(missing_entries)
    round_num = 0

    # Prepare verbose log
    if verbose:
        log_path = debug_file if debug_file else "utility_debug.txt"
        log_fh = open(log_path, "a")
        log_fh.write(f"\n=== strategy={strategy}  seed={seed}  missing_rate={missing_rate} ===\n")
        log_fh.write(f"{'Round':>6}  {'#Remaining':>10}  {'Mean':>10}  {'Min':>10}  {'Max':>10}  {'#Pos':>6}\n")
    else:
        log_fh = None

    while remaining:
        # Select candidate(s) to acquire
        if strategy == "uniform":
            to_acquire = [remaining[rng.integers(0, len(remaining))]]
        else:
            if strategy_base == "seu-us":
                candidates = select_candidates_us(remaining, n_sample, rng)
            else:
                candidates = select_candidates_es(remaining, model, X_masked, n_sample, rng)

            scores = [
                compute_seu_score(model, X_masked, y, i, j,
                                  utility=utility,
                                  acquisition_cost=acquisition_cost)
                for (i, j) in candidates
            ]

            # Verbose: log utility distribution for log_gain strategy
            if verbose and log_fh and utility == "log_gain":
                s_arr = np.array(scores)
                log_fh.write(
                    f"{round_num:>6}  {len(remaining):>10}  "
                    f"{s_arr.mean():>10.5f}  {s_arr.min():>10.5f}  "
                    f"{s_arr.max():>10.5f}  {(s_arr > 0).sum():>6}\n"
                )

            to_acquire = [candidates[int(np.argmax(scores))]]

        # Acquire: reveal true value from oracle
        for (i, j) in to_acquire:
            X_masked[i, j] = X_full[i, j]
            cumulative_cost += acquisition_cost
            remaining.remove((i, j))

        # Retrain on updated data
        model = NaiveBayesCategorical()
        model.fit(X_masked[train_idx], y[train_idx])
        acc = (model.predict(X_masked[val_idx]) == y[val_idx]).mean()

        cost_curve.append(cumulative_cost)
        accuracy_curve.append(acc)
        round_num += 1

    if log_fh:
        log_fh.close()

    return cost_curve, accuracy_curve

## Fully Observed Baseline

In [25]:
def run_fully_observed_baseline(X_full, y, seed=9999):
    """Train on fully observed data. Returns scalar accuracy (upper bound)."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(X_full.shape[0])
    split = int(0.8 * X_full.shape[0])
    train_idx, val_idx = idx[:split], idx[split:]
    model = NaiveBayesCategorical()
    model.fit(X_full[train_idx], y[train_idx])
    return (model.predict(X_full[val_idx]) == y[val_idx]).mean()